# Week 7: The Biology Gap Experiment\n\nIn Week 6, we found that **Regime 0** (the "Flat Line" regime) ignores Temperature.\nHypothesis: **This is the Biological Regime**.\n\n## Goal\nQuantify how much predictive power **Chlorophyll** adds.\n\n## Experiments\n1. **Global Comparison**: $R^2$ of `f(Phys)` vs `f(Phys + Bio)`.\n2. **Regime 0 Deep Dive**: Can we find a symbolic law involving Chlorophyll?

In [ ]:
import sys
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from pysr import PySRRegressor

sys.path.append(os.path.abspath('..'))

from scripts.preprocess import TRAIN_OUTPUT_PATH, TEST_OUTPUT_PATH

# 1. Load Data
print("Loading Data...")
ds_train = xr.open_dataset(TRAIN_OUTPUT_PATH)
ds_test = xr.open_dataset(TEST_OUTPUT_PATH)

df_train = ds_train.to_dataframe().reset_index().dropna()
df_test = ds_test.to_dataframe().reset_index().dropna()

# Sample for speed (RF is slow on 1M points)
N_SAMPLES = 50000
df_train_sub = df_train.sample(n=N_SAMPLES, random_state=42)

X_phys_train = df_train_sub[['sst', 'sss']].values
X_full_train = df_train_sub[['sst', 'sss', 'log_chl']].values
y_train = df_train_sub['fco2'].values

X_phys_test = df_test[['sst', 'sss']].values
X_full_test = df_test[['sst', 'sss', 'log_chl']].values
y_test = df_test['fco2'].values

# 2. Global RF Comparison
print("Training Physics-Only Model (SST, SSS)...")
rf_phys = RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
rf_phys.fit(X_phys_train, y_train)
r2_phys = r2_score(y_test, rf_phys.predict(X_phys_test))

print("Training Bio-Physics Model (SST, SSS, Chl)...")
rf_full = RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
rf_full.fit(X_full_train, y_train)
r2_full = r2_score(y_test, rf_full.predict(X_full_test))

print(f"\nGlobal Results (Test Set):")
print(f"Physics R2: {r2_phys:.4f}")
print(f"Bio+Phys R2: {r2_full:.4f}")
print(f"Gain:        {r2_full - r2_phys:.4f}")

In [ ]:
# 3. Regime 0 Deep Dive
# Let's re-identify Regime 0 using KMeans on Lat+Phys like in Week 6
print("\nIdentifying Regime 0...")
df_train_sub['lat_norm'] = df_train_sub['lat'] / 90.0
X_cluster = df_train_sub[['lat_norm', 'sst', 'sss', 'log_chl']].values
kmeans = KMeans(n_clusters=3, random_state=42).fit(X_cluster)
labels = kmeans.labels_

# Evaluate Improvement in EACH Regime
for k in range(3):
    mask = (labels == k)
    if np.sum(mask) < 100: continue
    
    X_p = X_phys_train[mask]
    X_f = X_full_train[mask]
    y_k = y_train[mask]
    
    # Quick Check on Training Subset
    r2_p_k = rf_phys.score(X_p, y_k)
    r2_f_k = rf_full.score(X_f, y_k)
    
    print(f"Regime {k}: Phys={r2_p_k:.3f}, Bio={r2_f_k:.3f}, Gain={r2_f_k - r2_p_k:.3f}")

print("\nWhich regime has the biggest gain? Let's assume it's Regime 0 (from previous order).")

In [ ]:
# 4. Symbolic Search with Biology in Regime 0
# We will use the regime with the largest Gain for this search
# (Assuming it persists as index 0, but let's just pick the max gain one)

gains = []
for k in range(3):
    mask = (labels == k)
    gain = rf_full.score(X_full_train[mask], y_train[mask]) - rf_phys.score(X_phys_train[mask], y_train[mask])
    gains.append(gain)

target_regime = np.argmax(gains)
print(f"\nSearching for Biological Law in Regime {target_regime} (Max Gain)...")

mask = (labels == target_regime)
X_sym = df_train_sub.loc[df_train_sub.index[mask], ['sst', 'sss', 'log_chl']].values
y_sym = y_train[mask]

# Sample further for PySR speed
if len(y_sym) > 2000:
    idx = np.random.choice(len(y_sym), 2000, replace=False)
    X_sym = X_sym[idx]
    y_sym = y_sym[idx]

model = PySRRegressor(
    niterations=30,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["exp"],
    verbosity=0
)
model.fit(X_sym, y_sym)

print(f"Best Bio-Physical Equation for Regime {target_regime}:")
print(model.sympy())

print("\nDoes it include x2 (log_chl)?")